# Training-side checks (`viz.ipynb`)

Reads the training records in `<paths.output>/<case>/`; trains nothing. Sections: training curves and probes, AE acceptance table, AE reconstruction slices (true 48³ vs decoded), latent distributions (test encodings vs DiT samples), memtest results.

- Every figure title names its data source; output of `selftest` (`configs/smoke.yaml`) is **synthetic data**.
- Headless: `python scripts/run_notebook.py notebooks/viz.ipynb --set DEVICE=cpu`, or `jobs/notebook_cpu.sub` / `notebook_gpu.sub`.
- AE reconstructions are cached per checkpoint step in `<case>/viz/cache/`, so a re-run only computes what is new; photon sums are recomputed from the cached field at the current threshold. Figures are saved to `<case>/viz/figs_viz/`.


In [ ]:
# parameters (run_notebook.py --set NAME=VALUE replaces these lines)
CONFIG = 'configs/v2.yaml'
CASE = 'v2a'
DEVICE = 'cpu'             # 'cuda' only speeds up the AE reconstruction cell
AE_CKPT = 'best'
N_RECON = 4                # val events in the AE reconstruction figure
FULL_RUN = 'full'          # generated latents for the latent histogram (None: skip)
FIG_SUBDIR = 'figs_viz'


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / 'sparseshower').is_dir() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from sparseshower.common import config
from sparseshower.loader import case_dir, meta_for
from sparseshower import viz
c = config(CONFIG)
CASE_DIR = case_dir(c, CASE)
FIG_DIR = CASE_DIR / 'viz' / FIG_SUBDIR
FIG_DIR.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    if fig is not None:
        fig.savefig(FIG_DIR / f'{name}.png', dpi=120, bbox_inches='tight')
        plt.show()
print('project  :', ROOT)
print('config   :', CONFIG, '| data', c['paths']['processed'])
print('case dir :', CASE_DIR, '(exists)' if CASE_DIR.exists() else '(MISSING)')
print('figures  ->', FIG_DIR)
if c.get('synthetic'):
    print('*** SYNTHETIC data (smoke config): nothing here is CORSIKA ***')


## 1. Training curves and probes

Left: training loss (thin: raw, thick: smoothed; struct also shows CE per level). Right: probes on fixed val events (AE: acceptance metrics, dotted lines are the thresholds; DiT/attr: denoising ratio R = |D − x|² / |σn|² at each σ, lower is better; struct: CE and expected child ratio at the first / last colour step, and lost children). The dashed black line is R_mean, the score that selects `<kind>_best.pt`.


In [ ]:
fig = viz.plot_training(c, CASE)
save(fig, '1_training')
for kind in ('ae', 'dit', 'struct', 'attr'):
    p = CASE_DIR / f'{kind}_done.json'
    mem = CASE_DIR / f'{kind}_memory.json'
    print(f"{kind:<7} done: {p.read_text().strip() if p.exists() else 'not finished'}")
    if mem.exists():
        print(f"        memory report: {mem.read_text().strip()[:300]}")


## 2. AE acceptance (`ae-eval` output)


In [ ]:
found = sorted((CASE_DIR / 'eval').glob('ae_*/passfail.md')) if (CASE_DIR / 'eval').exists() else []
for f in found:
    print(f'== {f.parent.name}')
    print(f.read_text())
if not found:
    print('no ae-eval output yet (jobs/ae_eval.sub)')


## 3. AE reconstruction slices

One row per val event: true 48³ photons (sum over y) | AE decode (photons on cells with p_occ > threshold) | decoded p_occ (max over y) | longitudinal profile.


In [ ]:
ids = [int(e) for e in meta_for(c)['split']['val']][:N_RECON]
if (CASE_DIR / f'ae_{AE_CKPT}.pt').exists():
    rows = viz.ae_recon(c, CASE, ids, AE_CKPT, DEVICE)
    save(viz.plot_ae_recon(c, rows, AE_CKPT), '3_ae_recon')
else:
    print(f'no ae_{AE_CKPT}.pt yet')


## 4. Latent distributions

Standardized test encodings μ (black) and DiT-generated latents from the `full` run (red), per channel. Overlap is necessary, not sufficient, for the DiT to have learned the latent distribution.


In [ ]:
save(viz.plot_latents(c, CASE, FULL_RUN), '4_latents')


## 5. memtest

Results of `jobs/memtest.sub` (`<paths.output>/memtest/*.json`): peak GPU memory and time of one training step per model, on the largest event in the cache.


In [ ]:
_ = viz.memtest_table(c)
